<a href="https://colab.research.google.com/github/AlanSojan15/ML_Lab/blob/Lab10/2547208_ML_Lab10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Implementation 1: High-Level API (Keras / tf.keras)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential

# 1. Dataset
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=np.float32)
y = np.array([[0], [1], [1], [0]], dtype=np.float32)

# 2. Model Architecture
model = Sequential(
    [
        Dense(4, input_dim=2, activation="tanh"),  # Hidden layer
        Dense(1, activation="sigmoid"),  # Output layer
    ]
)

# 3. Compilation
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.05),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

# 4. Training
history = model.fit(X, y, epochs=500, verbose=0)

# 5. Evaluation
predictions = model.predict(X)
print("Keras Predictions (Probabilities):\n", predictions)
print("Binary Predictions:\n", (predictions > 0.5).astype(int))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
Keras Predictions (Probabilities):
 [[4.3350854e-04]
 [9.9919748e-01]
 [9.9926078e-01]
 [9.8055694e-04]]
Binary Predictions:
 [[0]
 [1]
 [1]
 [0]]


##Implementation 2: Low-Level API (TensorFlow tf.GradientTape)

In [ ]:
import numpy as np
import tensorflow as tf

# 1. Dataset
X = tf.constant([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=tf.float32)
y = tf.constant([[0], [1], [1], [0]], dtype=tf.float32)

# 2. Model Parameters (Weights and Biases)
tf.random.set_seed(42)
W1 = tf.Variable(tf.random.normal([2, 4], stddev=0.5))
b1 = tf.Variable(tf.zeros([4]))
W2 = tf.Variable(tf.random.normal([4, 1], stddev=0.5))
b2 = tf.Variable(tf.zeros([1]))

optimizer = tf.optimizers.Adam(learning_rate=0.05)

# 3 & 4. Forward Pass and Training Loop
epochs = 500
for epoch in range(epochs):
    with tf.GradientTape() as tape:
        # Forward pass
        hidden = tf.nn.tanh(tf.matmul(X, W1) + b1)
        logits = tf.matmul(hidden, W2) + b2
        loss = tf.reduce_mean(
            tf.nn.sigmoid_cross_entropy_with_logits(labels=y, logits=logits)
        )

    # Compute and apply gradients
    gradients = tape.gradient(loss, [W1, b1, W2, b2])
    optimizer.apply_gradients(zip(gradients, [W1, b1, W2, b2]))

# 5. Evaluation
hidden_eval = tf.nn.tanh(tf.matmul(X, W1) + b1)
preds_prob = tf.nn.sigmoid(tf.matmul(hidden_eval, W2) + b2)

print("TensorFlow Low-Level Predictions:\n", preds_prob.numpy())
print("Binary Predictions:\n", tf.cast(preds_prob > 0.5, tf.int32).numpy())

TensorFlow Low-Level Predictions:
 [[1.1324399e-04]
 [9.9741417e-01]
 [9.9946606e-01]
 [2.3609924e-03]]
Binary Predictions:
 [[0]
 [1]
 [1]
 [0]]


##Hyperparameter Effects & AnalysisHidden Layer & Non-Linearity:
A single perceptron cannot solve XOR because it is not linearly separable. The hidden layer with non-linear activations ($\text{tanh}$ or $\text{ReLU}$) transforms the input space into a higher dimension where classes become separable.Activation Functions: $\text{Tanh}$ converges faster for the small XOR dataset because it is zero-centered. Standard $\text{ReLU}$ occasionally suffers from "dying ReLU" when all outputs get stuck at 0 with small architectures.Learning Rate: High rates ($0.05 - 0.1$) allow rapid convergence in $200-500$ epochs, whereas default rates like $0.001$ require several thousand epochs to escape saddle points.Loss Function: Binary Cross-Entropy provides steep gradients for confident wrong predictions, ensuring much faster convergence than Mean Squared Error.

##Conclusion
Both high-level (Keras) and low-level (TensorFlow) APIs successfully learn the XOR non-linear decision boundary using an MLP with at least 2 hidden units and non-linear activation. Keras abstracts gradient computation and layer management, while the low-level API exposes fine-grained parameter tracking via tf.GradientTape.